In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import cohen_kappa_score, roc_auc_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dl = pd.read_csv('data/data_dl.csv')

In [3]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [4]:
num_cols = data_dl.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = data_dl.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

In [5]:
cat_cols.remove('Target')
cat_cols

['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

In [6]:
# Ordinal encoding — preserve the natural order
label_map = {"Poor": 0, "Standard": 1, "Good": 2}
data_dl["Target_label"] = data_dl["Target"].map(label_map)

In [7]:
data_dl[data_dl.isnull().any(axis=1)]

,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,month3_balance,month3_credit_util,month3_delayed_payment,month3_delay_from_due_date,max_delayed_payment,max_credit_util,credit_util_var,balance_m1_m2,balance_m2_m3,Target_label
3,55,Entrepreneur,30689.89,2612.490833,2,5,4,100,4,9.0,...,419.880784,27.445422,6.0,5,9.0,41.154317,34.213323,-0.145042,0.056371,1
5,31,Lawyer,73928.46,5988.705000,4,5,8,0,8,7.0,...,633.080175,35.275437,7.0,7,10.0,42.769864,32.667415,0.560140,-0.193844,2
8,24,Doctor,114838.41,9843.867500,2,5,7,3,11,11.0,...,491.113388,26.114214,14.0,13,14.0,35.141567,9.717689,-0.061997,0.612714,1
9,45,Journalist,31370.80,NaN,1,6,12,2,-2,2.0,...,378.189779,25.502469,0.0,1,2.0,37.565053,20.705851,-0.190363,-0.107558,2
11,33,Engineer,88640.24,7266.686667,3,6,1,2,-1,NaN,...,195.369079,38.323639,0.0,4,2.0,41.036168,25.202502,0.098338,2.416203,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12490,38,Media_Manager,139664.96,11777.746667,3,6,12,4,14,12.0,...,182.547539,27.714375,12.0,14,12.0,40.171095,22.103331,NaN,2.482677,2
12492,48,_______,22620.79,NaN,6,2,9,0,27,15.0,...,254.733128,27.699504,18.0,27,19.0,37.450793,18.993999,-0.350503,0.553290,0
12495,19,Lawyer,42903.79,3468.315833,0,4,6,1,9,NaN,...,394.500158,35.549456,-1.0,14,0.0,35.716618,19.951316,0.272631,-0.110311,2
12496,45,Media_Manager,16680.35,NaN,1,1,5,4,1,0.0,...,233.301539,24.972853,NaN,1,0.0,41.212367,27.790639,-0.087474,0.460277,2


In [8]:
target_cols = ['Target_label']

In [9]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight= 'balanced',
    classes=np.array([0, 1, 2]),
    y=data_dl["Target_label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
# Pass into loss function — model pays more attention to rare classes
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [10]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [11]:
def null_fix(df):
    for col in df.columns:
        if col == "Monthly_Inhand_Salary" and df[col].isnull().any():
            df[col] = df['Annual_Income'] / 12
        elif df[col].isnull().any():
            df[col].fillna(0, inplace=True)
    return df

In [12]:
data_dl = data_dl.pipe(null_fix)

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_50730/904450771.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)


In [13]:
X_train, X_test , y_train , y_test = train_test_split(
    data_dl.drop(columns=target_cols),
    data_dl[target_cols],
    test_size=0.25,
    random_state=42,
    stratify=data_dl[target_cols]
)

In [14]:
print("Checking for NaN in training data...")
print(f"NaN in X_train: {X_train.isnull().sum().sum()}")
print(f"NaN in X_test: {X_test.isnull().sum().sum()}")
if X_train.isnull().any().any():
    print("\nColumns with NaN in train set:")
    print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
    # Fill NaN with median for numeric, mode for categorical
    for col in num_cols:
        if X_train[col].isnull().any():
            X_train[col].fillna(X_train[col].median(), inplace=True)
            X_test[col].fillna(X_test[col].median(), inplace=True)
    for col in cat_cols:
        if X_train[col].isnull().any():
            X_train[col].fillna(X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else "Unknown", inplace=True)
            X_test[col].fillna(X_test[col].mode()[0] if len(X_test[col].mode()) > 0 else "Unknown", inplace=True)
    print("NaN values filled.")

Checking for NaN in training data...
NaN in X_train: 0
NaN in X_test: 0


In [15]:
X_train.head()

,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,month2_delay_from_due_date,month3_balance,month3_credit_util,month3_delayed_payment,month3_delay_from_due_date,max_delayed_payment,max_credit_util,credit_util_var,balance_m1_m2,balance_m2_m3
3738,40,Musician,20877.19,1739.765833,7,6,20,5,20,20.0,...,20,292.372613,38.528879,17.0,17,21.0,38.528879,20.849362,-0.227512,0.040113
7274,40,Accountant,60015.00,5001.250000,7,7,24,2,14,0.0,...,14,542.316098,27.364181,12.0,14,18.0,39.142384,32.646463,0.555936,-0.299054
4810,29,Writer,19220.61,1601.717500,6,7,30,100,43,22.0,...,43,202.294673,33.173982,22.0,47,24.0,40.270504,30.038507,-0.123344,0.502679
121,37,Journalist,18717.02,1559.751667,8,5,15,5,4,18.0,...,7,274.900823,31.583289,16.0,9,20.0,33.813130,8.082557,0.056698,0.005579
603,33,Architect,122942.28,10245.190000,2,3,3,1,8,4.0,...,8,831.347622,32.334353,4.0,8,5.0,41.735281,17.316611,-0.638297,0.254215


In [16]:
label_encoder = LabelEncoder().fit(data_dl[cat_cols].values.ravel())
scaler = StandardScaler().fit(X_train[num_cols])

In [17]:
y = y_train['Target_label'].astype('category')

In [18]:
# Train one encoder per categorical column
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(data_dl[col].astype(str))         # fit on full data
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    le_dict[col] = le

# Now calculate cat_dims correctly
cat_dims = []
for col in cat_cols:
    vocab_size = len(le_dict[col].classes_) + 2 
    embedding_dim = min(8, (vocab_size + 1) // 2)
    cat_dims.append((vocab_size, embedding_dim))

print("cat_dims:", cat_dims)

cat_dims: [(18, 8), (6, 3), (5, 3), (9, 5)]


In [19]:
le_dict

{'Occupation': LabelEncoder(),
 'Credit_Mix': LabelEncoder(),
 'Payment_of_Min_Amount': LabelEncoder(),
 'Payment_Behaviour': LabelEncoder()}

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

In [21]:
class HistDataset(Dataset):
    def __init__(self, X, y, cat_cols, num_cols):
        # X already has integer-encoded cat columns — no re-encoding needed
        self.cat = torch.tensor(X[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(X[num_cols].values, dtype=torch.float32)
        y_vals = y.values.ravel() if hasattr(y, 'values') else np.array(y).ravel()
        self.y  = torch.tensor(y_vals, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat[idx], self.num[idx], self.y[idx]

In [22]:
# print(X_train[cat_cols].head()) 

In [23]:
train_set = HistDataset(X_train, y_train, cat_cols, num_cols)
val_set = HistDataset(X_test, y_test, cat_cols, num_cols)

In [24]:
# sample_weights = class_weights[y_train['Target_label'].values]

# sampler = torch.utils.data.WeightedRandomSampler(
#     weights=sample_weights,
#     num_samples=len(sample_weights),
#     replacement=True
# )

# train_loader = DataLoader(train_set, batch_size=512, sampler=sampler)

In [25]:
train_loader = DataLoader(train_set, batch_size= 512, shuffle = True)
val_loader = DataLoader(val_set, batch_size= 512, shuffle = False)

In [26]:
# cat_dims = []
# for col in cat_cols:
#     # Get unique values from the full dataset before split
#     vocab_size = len(data_dl[col].unique()) + 1  # +1 for safety
#     embedding_dim = min(50, (vocab_size + 1) // 2)
#     cat_dims.append((vocab_size, embedding_dim))

# print("cat_dims:", cat_dims)

In [27]:
# num_cols = [c for c in base_features if c not in mlp_cat_cols]


### MLP Classifier

In [28]:
class MLPClassifier(nn.Module):
    def __init__(self, cat_dims, num_cols, hidden = [128, 64, 32], dropout = 0.3):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(v, d) for v, d in cat_dims])
        in_d = sum(d for _, d in cat_dims) + len(num_cols)
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, 3))
        self.net = nn.Sequential(*layers)
        
    def forward(self, cx, nx):
        x = torch.cat([e(cx[:, i]) for i, e in enumerate(self.embs)] + [nx], dim=1)
        return self.net(x)

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [30]:
mlp_clf = MLPClassifier(cat_dims=cat_dims, num_cols=num_cols, hidden=[256, 128, 64], dropout=0.3)
mlp_clf.to(device)
print(f"Parameters : {sum(p.numel() for p in mlp_clf.parameters()):,}")
print(f"Cat embeds : {list(zip(cat_cols, [d for _, d in cat_dims]))}")

Parameters : 57,825
Cat embeds : [('Occupation', 8), ('Credit_Mix', 3), ('Payment_of_Min_Amount', 3), ('Payment_Behaviour', 5)]


In [31]:
class_weight_dl = torch.tensor(class_weights, dtype = torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight = class_weight_dl)

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_50730/2186993042.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  class_weight_dl = torch.tensor(class_weights, dtype = torch.float32).to(device)


In [32]:
optimizer = torch.optim.AdamW(mlp_clf.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

In [33]:
# # Check training data
# print("Train set - cat has NaN:", torch.isnan(torch.tensor(X_train[cat_cols].values)).any())
# print("Train set - num has NaN:", torch.isnan(torch.tensor(X_train[num_cols].values)).any())
# print("Train set - labels have NaN:", torch.isnan(torch.tensor(y_train.values)).any())

# # Check first batch
# cb, num_b, yb = next(iter(train_loader))
# print("First batch - cat has NaN:", torch.isnan(cb.float()).any())
# print("First batch - num has NaN:", torch.isnan(num_b.float()).any())
# print("First batch - labels have NaN:", torch.isnan(yb.float()).any())

In [34]:
def train_dl(model, train_loader, val_loader, criterion, optimizer, scheduler, device,
             max_epochs=300, patience=50):
    best_val_auc, best_state = 0.0, None
    patience_count = 0
    history = {'train_loss': [], 'val_auc': []}

    for epoch in range(max_epochs):
        model.train()
        ep_loss = 0.0
        nan_loss = False
        for cb, num_b, yb in train_loader:
            cb, num_b, yb = cb.to(device), num_b.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(cb, num_b), yb)
            loss.backward()
            # Inside training loop, after loss.backward():
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            ep_loss += loss.item()
            
            # Check for NaN loss
            if torch.isnan(loss):
                print("NaN loss detected! Stopping.")
                nan_loss = True
                break
        if nan_loss:
            break
        model.eval()
        vp, vt = [], []; nan_found = False
        with torch.no_grad():
            for cb, num_b, yb in val_loader:
                logits = model(cb.to(device), num_b.to(device))
                pred = torch.softmax(logits, dim=1).cpu()
                
                if torch.isnan(pred).any():
                    print(f"NaN in predictions at epoch {epoch}")
                    nan_found = True
                    break
                
                vp.append(pred)
                vt.append(yb)

        if nan_found or len(vp) == 0:
            print("Skipping AUC computation — NaN predictions.")
            continue   # skip to next epoch

        vp_np = torch.cat(vp).numpy()
        vt_np = torch.cat(vt).numpy()

        # For multiclass AUC, use the probability of the true class
        val_auc = roc_auc_score(vt_np, vp_np, multi_class='ovr')  # or 'ovo'
        # val_auc = roc_auc_score(vt_np, vp_np)
        scheduler.step(val_auc)

        history['train_loss'].append(ep_loss / len(train_loader))
        history['val_auc'].append(val_auc)

        if val_auc > best_val_auc:
            best_val_auc, patience_count = val_auc, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1

        if patience_count >= 50:
            print(f"Early stopping at epoch {epoch + 1}  |  Best Val AUC: {best_val_auc:.4f}")
            break
        if (epoch + 1) % 30 == 0:
            print(f"Epoch {epoch+1:3d}  loss: {ep_loss/len(train_loader):.4f}  val AUC: {val_auc:.4f}")

    model.load_state_dict(best_state)
    # gc.collect()
    return history

In [35]:
def evaluate_dl(model, val_loader, device):
    model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for cb, num_b, yb in val_loader:
            logits = model(cb.to(device), num_b.to(device))
            pred = torch.softmax(logits, dim=1).cpu()
            vp.append(pred)
            vt.append(yb)

    vp_np = torch.cat(vp).numpy()
    vt_np = torch.cat(vt).numpy()

    val_auc = roc_auc_score(vt_np, vp_np, multi_class='ovr')  # or 'ovo'
    report = classification_report(vt_np, vp_np.argmax(axis=1))
    cm = confusion_matrix(vt_np, vp_np.argmax(axis=1))
    print(f"Validation AUC: {val_auc:.4f}")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)
    return val_auc

In [36]:
mlp_train_result = train_dl(mlp_clf, train_loader, val_loader, criterion, optimizer, scheduler, device,
         max_epochs=300, patience=50)

Epoch  30  loss: 0.7869  val AUC: 0.7392
Epoch  60  loss: 0.7373  val AUC: 0.7357
Early stopping at epoch 67  |  Best Val AUC: 0.7424


In [37]:
mlp_eval_result = evaluate_dl(mlp_clf, val_loader, device)
mlp_eval_result

Validation AUC: 0.7424
Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.66      0.59       901
           1       0.70      0.42      0.53      1621
           2       0.42      0.73      0.53       603

    accuracy                           0.55      3125
   macro avg       0.55      0.60      0.55      3125
weighted avg       0.60      0.55      0.55      3125

Confusion Matrix:
[[592 161 148]
 [482 683 456]
 [ 34 129 440]]


0.7424288124402318

In [38]:
class LSTMClassifier(nn.Module):
    def __init__(self, cat_dims, num_cols, hidden = [128, 64, 32], dropout = 0.3):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size =sum(d for _, d in cat_dims) + len(num_cols), 
            hidden_size=hidden[0], batch_first=True)
        self.embs = nn.ModuleList([nn.Embedding(v, d) for v, d in cat_dims])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden[0], hidden[1])
        self.net = nn.Sequential(
            self.dropout,
            self.fc,
            nn.BatchNorm1d(hidden[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[1], hidden[2]),
            nn.BatchNorm1d(hidden[2]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[2], 3)
        )
        
        
    def forward(self, cx, nx):
        x = torch.cat([e(cx[:, i]) for i, e in enumerate(self.embs)] + [nx], dim=1)
        x = x.unsqueeze(1)  # Add sequence dimension
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Take last time step
        return self.net(lstm_out)

In [39]:
lstm_clf = LSTMClassifier(cat_dims=cat_dims, num_cols=num_cols, hidden=[128, 64, 32], dropout=0.3)
lstm_clf.to(device)
print(f"Parameters : {sum(p.numel() for p in lstm_clf.parameters()):,}")
print(f"Cat embeds : {list(zip(cat_cols, [d for _, d in cat_dims]))}")

Parameters : 107,617
Cat embeds : [('Occupation', 8), ('Credit_Mix', 3), ('Payment_of_Min_Amount', 3), ('Payment_Behaviour', 5)]


In [40]:
optimizer_lstm = torch.optim.AdamW(lstm_clf.parameters(), lr=0.001)
scheduler_lstm = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_lstm, patience=20, factor=0.5)
lstm_train_result = train_dl(lstm_clf, train_loader, val_loader, criterion,
                              optimizer_lstm, scheduler_lstm, device)

Epoch  30  loss: 0.8105  val AUC: 0.7398
Epoch  60  loss: 0.7773  val AUC: 0.7355
Early stopping at epoch 66  |  Best Val AUC: 0.7424


In [41]:
lstm_eval_result = evaluate_dl(lstm_clf, val_loader, device)    

Validation AUC: 0.7424
Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.68      0.60       901
           1       0.71      0.43      0.53      1621
           2       0.43      0.71      0.54       603

    accuracy                           0.56      3125
   macro avg       0.56      0.61      0.56      3125
weighted avg       0.60      0.56      0.55      3125

Confusion Matrix:
[[610 150 141]
 [491 696 434]
 [ 35 137 431]]


In [42]:
lstm_eval_result

0.7424489169557204

In [43]:
import shap
import torch
import torch.nn as nn
import numpy as np

class LSTMWrapper(nn.Module):
    """
    Fuses cat + num into a single float tensor so SHAP DeepExplainer
    can pass one background array through the model.
    Input shape: (batch, n_cat + n_num) — cat features first, num features after.
    """
    def __init__(self, lstm_clf, cat_cols, num_cols):
        super().__init__()
        self.model = lstm_clf
        self.n_cat = len(cat_cols)

    def forward(self, x):
        cx = x[:, :self.n_cat].long()          # integer-encoded categoricals
        nx = x[:, self.n_cat:].float()          # scaled numerics
        return self.model(cx, nx)



In [44]:
# Build the wrapper
wrapped_model = LSTMWrapper(lstm_clf, cat_cols, num_cols).to(device)
wrapped_model.eval()

LSTMWrapper(
  (model): LSTMClassifier(
    (lstm): LSTM(59, 128, batch_first=True)
    (embs): ModuleList(
      (0): Embedding(18, 8)
      (1): Embedding(6, 3)
      (2): Embedding(5, 3)
      (3): Embedding(9, 5)
    )
    (dropout): Dropout(p=0.3, inplace=False)
    (fc): Linear(in_features=128, out_features=64, bias=True)
    (net): Sequential(
      (0): Dropout(p=0.3, inplace=False)
      (1): Linear(in_features=128, out_features=64, bias=True)
      (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
      (4): Dropout(p=0.3, inplace=False)
      (5): Linear(in_features=64, out_features=32, bias=True)
      (6): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): ReLU()
      (8): Dropout(p=0.3, inplace=False)
      (9): Linear(in_features=32, out_features=3, bias=True)
    )
  )
)

In [ ]:
# Step 2 (FIXED) — Use GradientExplainer instead of DeepExplainer
# DeepExplainer breaks on BatchNorm1d + LSTM due to unsupported ops in the
# backprop graph. GradientExplainer uses gradient × input which works on any
# differentiable PyTorch model.

import shap
import torch
import numpy as np

# ── Background & test tensors (same as before) ───────────────────────────────
X_train_cat = torch.tensor(X_train[cat_cols].values, dtype=torch.float32)
X_train_num = torch.tensor(X_train[num_cols].values, dtype=torch.float32)
X_train_combined = torch.cat([X_train_cat, X_train_num], dim=1)

X_test_cat  = torch.tensor(X_test[cat_cols].values,  dtype=torch.float32)
X_test_num  = torch.tensor(X_test[num_cols].values,  dtype=torch.float32)
X_test_combined = torch.cat([X_test_cat, X_test_num], dim=1)

bg_idx     = np.random.choice(len(X_train_combined), size=200, replace=False)
background = X_train_combined[bg_idx].to(device)

# ── GradientExplainer ────────────────────────────────────────────────────────
# wrapped_model is the LSTMWrapper from Step 1 — no changes needed there
wrapped_model.eval()

explainer   = shap.GradientExplainer(wrapped_model, background)


Per-class SHAP shape: (572, 3)


In [75]:
# Correct extraction — sv is (n_samples, n_features, n_classes)
batch_size = 256
all_sv = []

X_test_gpu = X_test_combined.to(device)

for start in range(0, len(X_test_gpu), batch_size):
    batch = X_test_gpu[start : start + batch_size]
    sv    = explainer.shap_values(batch)          # (batch, n_features, n_classes)
    all_sv.append(sv)

shap_array = np.concatenate(all_sv, axis=0)       # (n_samples, n_features, n_classes)
print(f"shap_array shape: {shap_array.shape}")    # (572, 44, 3)

# Split into per-class list to keep consistent with downstream steps
# shap_values_fixed[c] = (n_samples, n_features)
shap_values_fixed = [shap_array[:, :, c] for c in range(3)]
print(f"Per-class shape: {shap_values_fixed[0].shape}")   # (572, 44)

# Feature importance
feature_names   = cat_cols + num_cols
mean_shap       = np.stack([np.abs(sv) for sv in shap_values_fixed]).mean(axis=0).mean(axis=0)
shap_importance = pd.Series(mean_shap, index=feature_names).sort_values(ascending=False)
print(shap_importance.head(10))

shap_array shape: (3125, 44, 3)
Per-class shape: (3125, 44)
Outstanding_Debt              0.204169
month3_delay_from_due_date    0.105413
Changed_Credit_Limit          0.102627
month2_delay_from_due_date    0.071542
Delay_from_due_date           0.052664
loan_type__personal_loan      0.035765
Credit_History_Age            0.034784
loan_type__student_loan       0.029044
loan_type__not_specified      0.027832
Credit_Utilization_Ratio      0.023878
dtype: float64


In [76]:
from torch import nn, optim

class TemperatureScaler(nn.Module):
    """Learns a single scalar T that scales logits before softmax."""
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, model, val_loader, device, lr=0.01, max_iter=50):
        """Fit T by minimising NLL on the validation set."""
        model.eval()
        nll = nn.CrossEntropyLoss()
        optimizer = optim.LBFGS([self.temperature], lr=lr, max_iter=max_iter)

        # Collect all logits and labels first
        logits_list, labels_list = [], []
        with torch.no_grad():
            for cb, num_b, yb in val_loader:
                cb, num_b = cb.to(device), num_b.to(device)
                logits_list.append(model(cb, num_b).cpu())
                labels_list.append(yb)

        all_logits = torch.cat(logits_list)
        all_labels = torch.cat(labels_list)

        def closure():
            optimizer.zero_grad()
            scaled = self.forward(all_logits)
            loss = nll(scaled, all_labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        print(f"Learned temperature T = {self.temperature.item():.4f}")
        return self

In [77]:

# Calibrate
temp_scaler = TemperatureScaler()
temp_scaler.fit(lstm_clf, val_loader, device)

# Get calibrated probabilities for the test set
def get_calibrated_probs(model, scaler, val_loader, device):
    model.eval(); scaler.eval()
    probs_list, labels_list = [], []
    with torch.no_grad():
        for cb, num_b, yb in val_loader:
            cb, num_b = cb.to(device), num_b.to(device)
            logits   = model(cb, num_b)
            scaled   = scaler(logits)
            probs    = torch.softmax(scaled, dim=1).cpu()
            probs_list.append(probs)
            labels_list.append(yb)
    return torch.cat(probs_list).numpy(), torch.cat(labels_list).numpy()

cal_probs, true_labels = get_calibrated_probs(lstm_clf, temp_scaler, val_loader, device)
# cal_probs shape: (n_test, 3)  — columns: [P(Poor), P(Standard), P(Good)]

Learned temperature T = 1.0740


In [78]:
cal_probs

array([[0.85348666, 0.14293951, 0.00357383],
       [0.56692857, 0.39694476, 0.03612663],
       [0.604718  , 0.33779407, 0.05748794],
       ...,
       [0.8261324 , 0.16881299, 0.00505459],
       [0.13794203, 0.32049122, 0.54156667],
       [0.24287796, 0.45873734, 0.2983847 ]],
      shape=(3125, 3), dtype=float32)

In [79]:
true_labels

array([0, 1, 1, ..., 0, 1, 1], shape=(3125,))

In [88]:
def build_risk_score(cal_probs, shap_values, feature_names):
    """
    Parameters
    ----------
    cal_probs    : np.ndarray (n, 3) — [P(Poor), P(Standard), P(Good)]
    shap_values  : list of 3 (n, f) arrays from DeepExplainer
    feature_names: list of str

    Returns
    -------
    pd.DataFrame with one row per sample
    """
    # P(Good) is the base for the score
    p_good = cal_probs[:, 2]
    p_standard = cal_probs[:, 1]

    # SHAP for the "Good" class only
    sv_good = shap_values[2]       # shape (n, n_features)

    # Scale raw probability → 300–850 scorecard range (credit-score convention)
    raw_score = 300 + (p_standard * 200) + (p_good * 350)

    # Build per-feature contribution columns (SHAP toward "Good")
    shap_df = pd.DataFrame(sv_good, columns=[f"shap_{f}" for f in feature_names])

    # Top 3 positive and negative contributors per row
    def top_contributors(row, n=3):
        sorted_idx = np.argsort(row.values)
        top_neg = [(feature_names[i], round(float(row.iloc[i]), 4))
                   for i in sorted_idx[:n]]
        top_pos = [(feature_names[i], round(float(row.iloc[i]), 4))
                   for i in sorted_idx[-n:][::-1]]
        return top_pos, top_neg

    pos_contribs, neg_contribs = zip(*[top_contributors(shap_df.iloc[i]) for i in range(len(shap_df))])

    # Risk band
    def score_to_band(s):
        if s >= 650:  return "Excellent"
        if s >= 550:  return "Good"
        if s >= 450:  return "Fair"
        return "Poor"

    result = pd.DataFrame({
        "p_poor":     cal_probs[:, 0].round(4),
        "p_standard": cal_probs[:, 1].round(4),
        "p_good":     cal_probs[:, 2].round(4),
        "risk_score": raw_score.round(0).astype(int),
        "risk_band":  [score_to_band(s) for s in raw_score],
        "top_positive_factors": [str(c) for c in pos_contribs],
        "top_negative_factors": [str(c) for c in neg_contribs],
    })

    return pd.concat([result, shap_df], axis=1)



In [89]:
scorecard_df = build_risk_score(cal_probs, shap_values_fixed, feature_names)
print(scorecard_df[["risk_score", "risk_band", "top_positive_factors", "top_negative_factors"]].head())

   risk_score risk_band                               top_positive_factors  \
0         330      Poor  [('loan_type__mortgage_loan', 0.1356), ('Credi...   
1         392      Poor  [('month3_delay_from_due_date', 0.1086), ('loa...   
2         388      Poor  [('loan_type__personal_loan', 0.0393), ('loan_...   
3         364      Poor  [('Changed_Credit_Limit', 0.3125), ('loan_type...   
4         505      Fair  [('Outstanding_Debt', 0.3093), ('Changed_Credi...   

                                top_negative_factors  
0  [('Changed_Credit_Limit', -0.5463), ('month3_d...  
1  [('max_credit_util', -0.0915), ('loan_type__cr...  
2  [('Outstanding_Debt', -0.2017), ('loan_type__a...  
3  [('month3_delay_from_due_date', -0.4178), ('mo...  
4  [('Credit_History_Age', -0.1042), ('loan_type_...  


In [90]:
scorecard_df.head(20)

,p_poor,p_standard,p_good,risk_score,risk_band,top_positive_factors,top_negative_factors,shap_Occupation,shap_Credit_Mix,shap_Payment_of_Min_Amount,...,shap_month2_delay_from_due_date,shap_month3_balance,shap_month3_credit_util,shap_month3_delayed_payment,shap_month3_delay_from_due_date,shap_max_delayed_payment,shap_max_credit_util,shap_credit_util_var,shap_balance_m1_m2,shap_balance_m2_m3
0,0.8535,0.1429,0.0036,330,Poor,"[('loan_type__mortgage_loan', 0.1356), ('Credi...","[('Changed_Credit_Limit', -0.5463), ('month3_d...",0.0,0.0,0.0,...,-0.367281,-0.021696,-0.135210,-0.001418,-0.519633,0.006192,0.061929,0.082212,0.0,0.0
1,0.5669,0.3969,0.0361,392,Poor,"[('month3_delay_from_due_date', 0.1086), ('loa...","[('max_credit_util', -0.0915), ('loan_type__cr...",0.0,0.0,0.0,...,0.063623,-0.036417,0.025242,0.006671,0.108646,0.002683,-0.091456,0.001711,0.0,0.0
2,0.6047,0.3378,0.0575,388,Poor,"[('loan_type__personal_loan', 0.0393), ('loan_...","[('Outstanding_Debt', -0.2017), ('loan_type__a...",0.0,0.0,0.0,...,0.014675,-0.027307,-0.018253,-0.000495,-0.023229,0.001283,0.009888,-0.003963,0.0,0.0
3,0.7040,0.2624,0.0336,364,Poor,"[('Changed_Credit_Limit', 0.3125), ('loan_type...","[('month3_delay_from_due_date', -0.4178), ('mo...",0.0,0.0,0.0,...,-0.306076,-0.006395,0.052901,0.000343,-0.417787,-0.003734,0.006704,0.001297,0.0,0.0
4,0.1824,0.5441,0.2735,505,Fair,"[('Outstanding_Debt', 0.3093), ('Changed_Credi...","[('Credit_History_Age', -0.1042), ('loan_type_...",0.0,0.0,0.0,...,0.031405,0.034941,-0.003207,-0.000582,0.031302,0.002378,0.034122,0.000741,0.0,0.0
5,0.7684,0.2175,0.0141,348,Poor,"[('month3_credit_util', 0.0883), ('loan_type__...","[('month3_delay_from_due_date', -0.5092), ('mo...",0.0,0.0,0.0,...,-0.335026,-0.010613,0.088332,0.003060,-0.509175,0.003143,-0.003597,0.000200,0.0,0.0
6,0.4046,0.5232,0.0723,430,Poor,"[('Outstanding_Debt', 0.4219), ('month3_delay_...","[('loan_type__auto_loan', -0.1365), ('Changed_...",0.0,0.0,0.0,...,0.068133,-0.033248,0.045457,-0.000479,0.098655,-0.003940,0.026003,-0.001980,0.0,0.0
7,0.1281,0.2334,0.6384,570,Good,"[('month3_delay_from_due_date', 0.1899), ('mon...","[('Changed_Credit_Limit', -0.0859), ('loan_typ...",0.0,0.0,0.0,...,0.130395,-0.000124,-0.019517,-0.002559,0.189893,0.000393,0.013010,0.003179,0.0,0.0
8,0.1860,0.4437,0.3703,518,Fair,"[('month3_balance', 0.1752), ('max_credit_util...","[('month2_credit_util', -0.0503), ('month3_cre...",0.0,0.0,0.0,...,-0.014819,0.175157,-0.049484,0.000149,-0.018485,0.001081,0.073777,-0.000392,0.0,0.0
9,0.4469,0.3574,0.1957,440,Poor,"[('Changed_Credit_Limit', 0.7562), ('month3_cr...","[('Outstanding_Debt', -0.3189), ('month3_delay...",0.0,0.0,0.0,...,-0.096235,-0.003279,0.087179,-0.003475,-0.173514,0.002310,-0.006306,0.003161,0.0,0.0


In [95]:
scorecard_df['risk_score'].median()

np.float64(505.0)

In [97]:
import joblib
import os

os.makedirs("artefacts", exist_ok=True)

# Model weights
torch.save(lstm_clf.state_dict(),   "artefacts/lstm_clf.pt")
torch.save(temp_scaler.state_dict(),"artefacts/temp_scaler.pt")

# Preprocessors
joblib.dump(scaler,   "artefacts/scaler.pkl")
joblib.dump(le_dict,  "artefacts/le_dict.pkl")

# SHAP explainer background (needed to re-init DeepExplainer)
np.save("artefacts/shap_background.npy", background.cpu().numpy())

# Save cat_dims and column lists (needed to rebuild model)
import json
meta = {
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "cat_dims": cat_dims,   # list of [vocab_size, emb_dim] pairs
}
meta["tfidf_prefix"] = "loan_type__"   # add this before json.dump(meta, f)
with open("artefacts/meta.json", "w") as f:
    json.dump(meta, f)

print("Artefacts saved.")

Artefacts saved.
